In [0]:
from pyspark.sql import functions as F

RAW = "/Volumes/workspace/nightshift/raw_logs"
CHK = f"{RAW}/_checkpoints"
DB = "workspace.nightshift"

# Run logs: Auto Loader picks up only new files each run
(spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", f"{CHK}/runs_schema")
    .load(f"{RAW}/runs/")
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingested_at", F.current_timestamp())
    .writeStream
    .option("checkpointLocation", f"{CHK}/runs")
    .trigger(availableNow=True)
    .toTable(f"{DB}.bronze_runs")
    .awaitTermination())

# Small reference files: plain batch loads
spark.read.option("header", True).csv(f"{RAW}/benches.csv").write.mode("overwrite").saveAsTable(f"{DB}.bronze_benches")
spark.read.option("multiLine", True).json(f"{RAW}/builds.json").write.mode("overwrite").saveAsTable(f"{DB}.bronze_builds")
spark.read.option("header", True).csv(f"{RAW}/ground_truth.csv").write.mode("overwrite").saveAsTable(f"{DB}.bronze_ground_truth")

In [0]:
%sql
SELECT event, count(*) FROM workspace.nightshift.bronze_runs GROUP BY event

In [0]:
%sql
SELECT status, count(*) AS n FROM workspace.nightshift.silver_test_results GROUP BY status
UNION ALL
SELECT 'REBOOTS', count(*) FROM workspace.nightshift.silver_watchdog_events

In [0]:
%sql
SELECT test_name, avg_fail_rate, builds_intermittent
FROM workspace.nightshift.gold_test_flakiness
WHERE is_flaky;

In [0]:
%sql
SELECT * FROM workspace.nightshift.gold_build_regressions;

In [0]:
%sql
SELECT night, pass_rate, reboots FROM workspace.nightshift.gold_bench_nightly_health WHERE bench_id = 'DUO-03' ORDER BY night;